# Developer Automation System using MCP Protocol
**Case Study — Developer Tools Domain**

This Colab notebook demonstrates a protocol-driven developer productivity system using the **Model Context Protocol (MCP)**.

### What this prototype does
- Runs an MCP **server** that exposes developer tools.
- Connects an MCP **client** to the server using standard client/server communication over stdio.
- Supports multiple tools: list/read/write files, analyze Python code safely with `ast`, and run a small automation task.
- Demonstrates tool discovery and tool invocation through MCP rather than calling Python functions directly.

**Safety note:** The demo works only inside `/content/mcp_workspace`. The code-analysis tool parses code but does not execute submitted code. The automation tool creates a report; it does not run shell commands.

## 1. Install the MCP SDK
Run this cell first. If Colab asks to restart the runtime, restart it and continue from the next cell.

In [1]:
%pip -q install "mcp>=1.9,<2.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.7 MB/s eta 0:00:00


## 2. Create a workspace and sample files

In [2]:
from pathlib import Path
import sys, os, textwrap, json, asyncio

WORKSPACE = Path("/content/mcp_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)

(WORKSPACE / "sample.py").write_text(textwrap.dedent('''
    import os

    def calculate_total(prices):
        total = 0
        for price in prices:
            total += price
        return total

    def greet(name):
        print("Hello", name)
''').strip() + "\n", encoding="utf-8")

(WORKSPACE / "README.md").write_text(
    "# Sample Developer Project\n\nA tiny project used to demonstrate MCP file and code tools.\n",
    encoding="utf-8"
)

print("Workspace:", WORKSPACE)
print("Initial files:", sorted(p.name for p in WORKSPACE.iterdir()))

Workspace: /content/mcp_workspace
Initial files: ['README.md', 'sample.py']


## 3. Implement the MCP server

The server exposes five tools:
1. `list_files` — list files in the workspace.
2. `read_file` — read a text file.
3. `write_file` — create or update a text file.
4. `analyze_python` — report functions, imports, and simple code-quality observations using Python's AST parser.
5. `create_automation_report` — create a project inventory report.

All file paths are validated to remain within the workspace.

In [5]:
server_code = r"""
from pathlib import Path
import ast
from mcp.server.fastmcp import FastMCP

WORKSPACE = Path("/content/mcp_workspace").resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
mcp = FastMCP("Developer Automation MCP Server")

def safe_path(relative_path: str) -> Path:
    candidate = (WORKSPACE / relative_path).resolve()
    if candidate != WORKSPACE and WORKSPACE not in candidate.parents:
        raise ValueError("Path must remain inside the MCP workspace.")
    return candidate

@mcp.tool()
def list_files() -> dict:
    '''List files in the developer workspace.'''
    files = []
    for p in sorted(WORKSPACE.rglob("*")):
        if p.is_file():
            files.append(str(p.relative_to(WORKSPACE)))
    return {"workspace": str(WORKSPACE), "files": files, "count": len(files)}

@mcp.tool()
def read_file(path: str) -> dict:
    '''Read a UTF-8 text file from the workspace.'''
    p = safe_path(path)
    if not p.is_file():
        return {"ok": False, "error": "File not found", "path": path}
    try:
        return {"ok": True, "path": path, "content": p.read_text(encoding="utf-8")}
    except UnicodeDecodeError:
        return {"ok": False, "error": "File is not UTF-8 text", "path": path}

@mcp.tool()
def write_file(path: str, content: str) -> dict:
    '''Create or replace a UTF-8 text file inside the workspace.'''
    p = safe_path(path)
    if p.exists() and p.is_dir():
        return {"ok": False, "error": "Path is a directory"}
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return {"ok": True, "path": str(p.relative_to(WORKSPACE)), "characters_written": len(content)}

@mcp.tool()
def analyze_python(path: str) -> dict:
    '''Statically analyze a Python file; submitted code is never executed.'''
    p = safe_path(path)
    if not p.is_file():
        return {"ok": False, "error": "File not found", "path": path}
    source = p.read_text(encoding="utf-8")
    try:
        tree = ast.parse(source, filename=path)
    except SyntaxError as e:
        return {"ok": False, "error": "SyntaxError", "line": e.lineno, "message": e.msg}
    functions, classes, imports, issues = [], [], [], []
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            functions.append({"name": node.name, "line": node.lineno, "args": len(node.args.args)})
            if ast.get_docstring(node) is None:
                issues.append({"line": node.lineno, "type": "missing_docstring", "name": node.name})
        elif isinstance(node, ast.ClassDef):
            classes.append({"name": node.name, "line": node.lineno})
        elif isinstance(node, ast.Import):
            imports.extend(alias.name for alias in node.names)
        elif isinstance(node, ast.ImportFrom):
            imports.append(node.module or "")
    return {
        "ok": True, "path": path, "line_count": len(source.splitlines()),
        "functions": functions, "classes": classes, "imports": sorted(set(imports)),
        "observations": issues,
        "note": "Static AST analysis only; code was not executed."
    }

@mcp.tool()
def create_automation_report(report_name: str = "automation_report.md") -> dict:
    '''Generate a Markdown inventory report for the workspace.'''
    report_path = safe_path(report_name)
    files = sorted(p for p in WORKSPACE.rglob("*") if p.is_file() and p != report_path)
    lines = ["# Developer Automation Report", "", f"Files inventoried: {len(files)}", ""]
    for p in files:
        try:
            size = p.stat().st_size
        except OSError:
            size = 0
        lines.append(f"- `{p.relative_to(WORKSPACE)}` — {size} bytes")
    report_path.parent.mkdir(parents=True, exist_ok=True)
    report_path.write_text("\\n".join(lines) + "\\n", encoding="utf-8")
    return {"ok": True, "report": str(report_path.relative_to(WORKSPACE)), "files_included": len(files)}

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

server_path = Path("/content/mcp_server.py")
server_path.write_text(server_code, encoding="utf-8")
print("MCP server written to:", server_path)
print("Server tools defined: list_files, read_file, write_file, analyze_python, create_automation_report")

MCP server written to: /content/mcp_server.py
Server tools defined: list_files, read_file, write_file, analyze_python, create_automation_report


## 4. Connect the MCP client to the server

The client launches the server as a subprocess and communicates using MCP's stdio transport. It first discovers available tools, then calls them through `ClientSession`.

In [7]:

import sys
import os
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

server_params = StdioServerParameters(
    command=sys.executable,
    args=["-u", "/content/mcp_server.py"],
    env={**os.environ, "PYTHONUNBUFFERED": "1"}
)

# Use the original stderr stream instead of Colab's wrapped stderr.
stdio_context = stdio_client(
    server_params,
    errlog=sys.__stderr__
)

read_stream, write_stream = await stdio_context.__aenter__()

session_context = ClientSession(read_stream, write_stream)
session = await session_context.__aenter__()

await session.initialize()

tool_result = await session.list_tools()

print("MCP connection initialized successfully.")
print("Tools discovered from server:")

for tool in tool_result.tools:
    print(f"- {tool.name}: {tool.description}")


MCP connection initialized successfully.
Tools discovered from server:
- list_files: List files in the developer workspace.
- read_file: Read a UTF-8 text file from the workspace.
- write_file: Create or replace a UTF-8 text file inside the workspace.
- analyze_python: Statically analyze a Python file; submitted code is never executed.
- create_automation_report: Generate a Markdown inventory report for the workspace.


## 5. Execute multiple developer tools through MCP

In [8]:
# 1) List files using the MCP server
result = await session.call_tool("list_files", arguments={})
print("LIST FILES")
print(result.content[0].text)

# 2) Read the sample source file through MCP
result = await session.call_tool("read_file", arguments={"path": "sample.py"})
print("\\nREAD FILE")
print(result.content[0].text)

# 3) Run static code analysis through MCP
result = await session.call_tool("analyze_python", arguments={"path": "sample.py"})
print("\\nCODE ANALYSIS")
print(result.content[0].text)

# 4) Write a new file through MCP
result = await session.call_tool(
    "write_file",
    arguments={
        "path": "notes.txt",
        "content": "Created by the Developer Automation System via an MCP tool call.\\n"
    }
)
print("\\nWRITE FILE")
print(result.content[0].text)

# 5) Trigger an automation task through MCP
result = await session.call_tool(
    "create_automation_report",
    arguments={"report_name": "reports/project_inventory.md"}
)
print("\\nAUTOMATION REPORT")
print(result.content[0].text)

LIST FILES
{
  "workspace": "/content/mcp_workspace",
  "files": [
    "README.md",
    "sample.py"
  ],
  "count": 2
}
\nREAD FILE
{
  "ok": true,
  "path": "sample.py",
  "content": "import os\n\ndef calculate_total(prices):\n    total = 0\n    for price in prices:\n        total += price\n    return total\n\ndef greet(name):\n    print(\"Hello\", name)\n"
}
\nCODE ANALYSIS
{
  "ok": true,
  "path": "sample.py",
  "line_count": 10,
  "functions": [
    {
      "name": "calculate_total",
      "line": 3,
      "args": 1
    },
    {
      "name": "greet",
      "line": 9,
      "args": 1
    }
  ],
  "classes": [],
  "imports": [
    "os"
  ],
  "observations": [
    {
      "line": 3,
      "type": "missing_docstring",
      "name": "calculate_total"
    },
    {
      "line": 9,
      "type": "missing_docstring",
      "name": "greet"
    }
  ],
  "note": "Static AST analysis only; code was not executed."
}
\nWRITE FILE
{
  "ok": true,
  "path": "notes.txt",
  "characters_written": 

## 6. Verify the generated report

In [9]:
result = await session.call_tool(
    "read_file",
    arguments={"path": "reports/project_inventory.md"}
)
print(result.content[0].text)

{
  "ok": true,
  "path": "reports/project_inventory.md",
  "content": "# Developer Automation Report\\n\\nFiles inventoried: 3\\n\\n- `README.md` — 88 bytes\\n- `notes.txt` — 66 bytes\\n- `sample.py` — 162 bytes\\n"
}


## 7. Test path validation

This negative test attempts to access a path outside the workspace. It should return an error rather than read the requested file.

In [10]:
result = await session.call_tool(
    "read_file",
    arguments={"path": "../../etc/passwd"}
)
print(result.content[0].text)

Error executing tool read_file: Path must remain inside the MCP workspace.


## 8. Close the MCP client and server cleanly

Run this cell when you have finished using the MCP session.

In [12]:

import sys
import os
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_mcp_demo():
    server_params = StdioServerParameters(
        command=sys.executable,
        args=["-u", "/content/mcp_server.py"],
        env={**os.environ, "PYTHONUNBUFFERED": "1"}
    )

    async with stdio_client(
        server_params,
        errlog=sys.__stderr__
    ) as (read_stream, write_stream):

        async with ClientSession(
            read_stream, write_stream
        ) as session:

            await session.initialize()

            tools = await session.list_tools()
            print("MCP connection initialized.")

            print("\nAvailable tools:")
            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")

            # Execute sample tool calls while the session is active.
            for tool_name, arguments in [
                ("list_files", {}),
                ("analyze_python", {"path": "sample.py"}),
                ("create_automation_report",
                 {"report_name": "reports/project_inventory.md"})
            ]:
                print(f"\n--- {tool_name} ---")
                result = await session.call_tool(
                    tool_name, arguments=arguments
                )

                for item in result.content:
                    if hasattr(item, "text"):
                        print(item.text)

            print("\nTool execution completed.")

    print("\nMCP session and transport closed cleanly.")

await run_mcp_demo()


MCP connection initialized.

Available tools:
- list_files: List files in the developer workspace.
- read_file: Read a UTF-8 text file from the workspace.
- write_file: Create or replace a UTF-8 text file inside the workspace.
- analyze_python: Statically analyze a Python file; submitted code is never executed.
- create_automation_report: Generate a Markdown inventory report for the workspace.

--- list_files ---
{
  "workspace": "/content/mcp_workspace",
  "files": [
    "README.md",
    "notes.txt",
    "reports/project_inventory.md",
    "sample.py"
  ],
  "count": 4
}

--- analyze_python ---
{
  "ok": true,
  "path": "sample.py",
  "line_count": 10,
  "functions": [
    {
      "name": "calculate_total",
      "line": 3,
      "args": 1
    },
    {
      "name": "greet",
      "line": 9,
      "args": 1
    }
  ],
  "classes": [],
  "imports": [
    "os"
  ],
  "observations": [
    {
      "line": 3,
      "type": "missing_docstring",
      "name": "calculate_total"
    },
    {


## 9. Architecture and workflow

```text
Developer / Notebook
        |
        v
   MCP Client
   - initialize session
   - discover tools
   - call tools with arguments
        |
        | MCP over stdio
        v
   MCP Server
   - list_files
   - read_file / write_file
   - analyze_python (AST)
   - create_automation_report
        |
        v
 /content/mcp_workspace
```

### Conclusion
This prototype demonstrates protocol-driven integration of developer tools. The client discovers server capabilities and invokes multiple tools through MCP, while the server centralizes tool logic and workspace validation. It can be extended with Git operations, test-result summaries, issue tracker integrations, or an approval step for sensitive actions.

### Limitations
- The workspace is temporary Colab storage and may disappear when the runtime resets.
- This is a local stdio client/server demonstration, not a remotely hosted network service.
- Code analysis is static and intentionally does not execute source files.
- Production systems should add authentication/authorization for remote transports, structured logging, rate limits, tests, and explicit approval for destructive operations.